# Análisis Intermareal con Sentinel-2 SCL — Versión Refactorizada

Pipeline modular para análisis de zonas intermareales usando Scene Classification Layer (SCL) de Sentinel-2 L2A.

**Arquitectura**: Usa el paquete `intertidal` con 7 módulos especializados.

## 1. Instalación e Imports

In [1]:
# Instalar dependencias (solo primera vez)
%pip install openeo xarray geopandas rioxarray matplotlib contextily rasterio pyproj shapely scikit-image netcdf4 scipy --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paquete intertidal modular
from intertidal import (
    GeometryProcessor,
    RasterProcessor,
    OpenEOClient,
    SCLProcessor,
    IntertidalMapper,
    TideAnalyzer,
    Visualizer
)

# Modelos de marea
from tidemodel import PyTMDTideModel, CopernicusTideModel

print("✅ Imports completados")

✅ Imports completados


c:\Users\Jorge\sketch_fitton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración del Análisis

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────────────────────────

site = "Gijón"
time_extent = ["2024-01-01", "2024-12-31"]
reference_time_extent = ["2024-01-01", "2024-12-31"]  # Para reference map

# Fechas de muestra para análisis exploratorio
selected_dates = [
    "2024-01-04", "2024-03-14", "2024-05-28", "2024-07-02",
    "2024-08-11", "2024-09-05", "2024-10-30", "2024-12-09"
]

# Umbrales de calidad
scl_max_bad_fraction = 0.20  # 20% píxeles malos para filtro global
scl_transition_max_bad_fraction = 0.20  # 20% para filtro en transición

# Directorios de salida
rgb_dir = "tifs_rgb"
scl_dir = "tifs_scl"
os.makedirs(rgb_dir, exist_ok=True)
os.makedirs(scl_dir, exist_ok=True)

# Clases SCL malas (nubes, sombras, nieve)
scl_bad_classes = [3, 8, 9, 10, 11]

# AOI en coordenadas DMS (grados, minutos, segundos)
aoi_dms_list = [
    "5°39'27.89\"W 43°33'15.40\"N",
    "5°38'33.30\"W 43°33'13.10\"N",
    "5°38'36.10\"W 43°32'40.90\"N",
    "5°39'30.70\"W 43°32'43.20\"N",
]

print(f"✅ Configuración: {site} | {time_extent[0]} - {time_extent[1]}")

✅ Configuración: Gijón | 2024-01-01 - 2024-12-31


## 3. Definir AOI y Conectar a OpenEO

In [4]:
# Crear polígono AOI y bbox
geo = GeometryProcessor()
aoi_polygon = geo.make_polygon(aoi_dms_list)
aoi_bbox = geo.bbox_from_polygon(aoi_polygon)

print(f"AOI bbox: {aoi_bbox}")
print(f"AOI área: {aoi_polygon.area:.6f} grados²")

# Conectar a OpenEO Copernicus Dataspace
client = OpenEOClient(backend_url="openeo.dataspace.copernicus.eu")
connection = client.connect(use_oidc=True)

print("✅ Conectado a OpenEO")

ValueError: No se puede parsear: 5°39'27.89"W 43°33'15.40"N

## 4. Consultar Fechas Disponibles

In [ ]:
# Obtener todas las fechas disponibles con baja nubosidad
available_dates = client.get_available_dates(
    bbox=aoi_bbox,
    time_extent=time_extent,
    max_cloud_cover=80  # Búsqueda permisiva, filtraremos luego por SCL
)

print(f"\n📅 Fechas disponibles en {time_extent[0]} - {time_extent[1]}: {len(available_dates)}")
print(f"Primeras 10: {available_dates[:10]}")

## 5. Descargar Datos RGB y SCL

In [ ]:
# Descargar RGB para fechas seleccionadas
print("\n📥 Descargando RGB...")
rgb_results = client.download_rgb_batch(selected_dates, aoi_bbox, rgb_dir)
print(f"RGB descargados: {len([r for r in rgb_results.values() if r == 'ok'])}/{len(selected_dates)}")

# Descargar SCL para fechas seleccionadas
print("\n📥 Descargando SCL...")
scl_results = client.download_scl_batch(selected_dates, aoi_bbox, scl_dir)
print(f"SCL descargados: {len([r for r in scl_results.values() if r == 'ok'])}/{len(selected_dates)}")

print("\n✅ Descargas completadas")

## 6. Análisis de Calidad SCL (Filtro Global)

In [ ]:
# Inicializar procesador SCL
scl_processor = SCLProcessor(bad_classes=scl_bad_classes)

# Filtrar fechas por calidad global
quality_result = scl_processor.filter_dates_by_quality(
    selected_dates,
    scl_dir,
    max_bad_fraction=scl_max_bad_fraction
)

clean_dates = quality_result['valid']
discarded_dates = quality_result['discarded']
scl_stats = quality_result['stats']

print(f"\n📊 Análisis de Calidad SCL (filtro global):")
print(f"   Fechas válidas: {len(clean_dates)}")
print(f"   Fechas descartadas: {len(discarded_dates)}")
print(f"\n   Fechas válidas: {clean_dates}")
if discarded_dates:
    print(f"   Descartadas: {discarded_dates}")

## 7. Visualizar RGB y SCL

In [ ]:
# Visualizar algunas fechas limpias
for date in clean_dates[:4]:  # Primeras 4 fechas
    Visualizer.plot_rgb_with_scl(
        date,
        rgb_dir,
        scl_dir,
        polygon=aoi_polygon,
        scl_stats=scl_stats,
        scl_max_bad_fraction=scl_max_bad_fraction
    )

## 8. Construir Reference Map Local

In [ ]:
# Cargar stack de SCLs limpios
print(f"\n Cargando stack de {len(clean_dates)} fechas limpias...")
scl_stack = scl_processor.load_stack(clean_dates, scl_dir)
print(f"Stack shape: {scl_stack.shape}")

# Construir reference map
print("\n Construyendo reference map local...")
reference_map, P_water, P_land = scl_processor.build_reference_map_local(
    scl_stack,
    stable_threshold=0.98,
    coastal_buffer_pixels=20
)

# Estadísticas
transition_pixels = (reference_map == 0).sum()
water_pixels = (reference_map == 1).sum()
land_pixels = (reference_map == 2).sum()

print(f"\n📊 Reference Map:")
print(f"   Transición (intermareal): {transition_pixels:,} píxeles")
print(f"   Agua estable: {water_pixels:,} píxeles")
print(f"   Tierra estable: {land_pixels:,} píxeles")

# Visualizar
Visualizer.plot_reference_map(reference_map)

## 9. Filtro de Calidad por Zona de Transición

Evaluar nubosidad solo en la zona intermareal para recuperar fechas descartadas por el filtro global.

In [ ]:
# Máscara de zona de transición
transition_mask = (reference_map == 0)

# Evaluar todas las fechas descargadas
transition_stats = {}
transition_valid = []
transition_discarded = []

for date in selected_dates:
    scl_path = os.path.join(scl_dir, f"scl_{date}.tif")
    if not os.path.exists(scl_path):
        continue
    
    scl_arr = RasterProcessor.read_scl(scl_path)
    stats = scl_processor.compute_transition_stats(scl_arr, transition_mask)
    transition_stats[date] = stats
    
    if stats['bad_fraction'] <= scl_transition_max_bad_fraction:
        transition_valid.append(date)
    else:
        transition_discarded.append(date)

# Fechas recuperadas: estaban en descartadas global pero pasan filtro transición
recovered = set(transition_valid) - set(clean_dates)

print(f"\n Filtro por Zona de Transición:")
print(f"   Fechas válidas (transición): {len(transition_valid)}")
print(f"   Fechas descartadas: {len(transition_discarded)}")
print(f"   Fechas RECUPERADAS: {len(recovered)}")
if recovered:
    print(f"   Recuperadas: {sorted(recovered)}")

## 10. Análisis de Marea

In [ ]:
# Obtener centroide de agua para predicción de marea
import rasterio
with rasterio.open(os.path.join(scl_dir, f"scl_{clean_dates[0]}.tif")) as src:
    transform = src.transform
    crs = src.crs

mapper = IntertidalMapper(client)
lon_centroid, lat_centroid = mapper.get_water_centroid(reference_map, transform)

print(f"\n Centroide de agua estable: ({lon_centroid:.4f}, {lat_centroid:.4f})")

# Predecir mareas usando GOT4.10
tide_model = PyTMDTideModel(
    model_name="GOT4.10c",
    model_dir="tide_models/GOT4.10c"
)

# Crear DataFrame con fechas
df_tides = pd.DataFrame({
    'date': pd.to_datetime(transition_valid)
})

# Calcular alturas de marea para cada fecha (a las 11:00 UTC)
tide_heights = []
for date in df_tides['date']:
    dt = date.replace(hour=11, minute=0, second=0)
    height = tide_model.predict_tide_height(
        lon=lon_centroid,
        lat=lat_centroid,
        datetime=dt
    )
    tide_heights.append(height)

df_tides['tide_height_m'] = tide_heights

print(f"\n📊 Estadísticas de marea:")
print(df_tides[['date', 'tide_height_m']].describe())

## 11. Visualizar Serie Temporal de Marea

In [ ]:
# Visualizar serie temporal
Visualizer.plot_tide_timeseries(df_tides, site=site)

## 12. Validación con Mareógrafo (Opcional)

In [ ]:
# Si existe archivo de mareógrafo, cargar y comparar
gauge_file = "mareografo_gijon_2024.xlsx"

if os.path.exists(gauge_file):
    analyzer = TideAnalyzer()
    
    # Cargar datos observados
    gauge_data = analyzer.load_gauge_data(
        gauge_file,
        date_col="fecha",
        height_col="nivel_m"
    )
    
    # Merge con predicciones
    df_comparison = pd.merge(
        gauge_data.rename(columns={'datetime': 'date', 'height_m': 'observed_height_m'}),
        df_tides.rename(columns={'tide_height_m': 'got410_height_m'}),
        on='date',
        how='outer'
    )
    
    # Calcular métricas
    metrics = TideAnalyzer.calculate_metrics(
        df_comparison,
        cmems_available=False,
        obs_available=True
    )
    
    # Imprimir métricas
    TideAnalyzer.print_metrics(metrics)
    
    # Visualizar comparación
    Visualizer.plot_tide_model_comparison(df_comparison, site=site)
else:
    print("⚠️ No se encontró archivo de mareógrafo para validación")

## 13. Resumen Final

In [ ]:
print("\n" + "="*70)
print(f"📋 RESUMEN DEL ANÁLISIS — {site}")
print("="*70)
print(f"\n🗓️ Periodo analizado: {time_extent[0]} → {time_extent[1]}")
print(f"\n📅 Fechas:")
print(f"   Disponibles en backend: {len(available_dates)}")
print(f"   Descargadas: {len(selected_dates)}")
print(f"   Válidas (filtro global): {len(clean_dates)}")
print(f"   Válidas (filtro transición): {len(transition_valid)}")
print(f"   Recuperadas por filtro transición: {len(recovered)}")
print(f"\n🗺️ Reference Map:")
print(f"   Zona intermareal: {transition_pixels:,} píxeles ({transition_pixels/(reference_map.size)*100:.1f}%)")
print(f"   Agua estable: {water_pixels:,} píxeles ({water_pixels/(reference_map.size)*100:.1f}%)")
print(f"   Tierra estable: {land_pixels:,} píxeles ({land_pixels/(reference_map.size)*100:.1f}%)")
print(f"\n🌊 Marea (GOT4.10c):")
print(f"   Altura media: {df_tides['tide_height_m'].mean():.2f} m")
print(f"   Rango: {df_tides['tide_height_m'].min():.2f} - {df_tides['tide_height_m'].max():.2f} m")
print(f"   Amplitud: {df_tides['tide_height_m'].max() - df_tides['tide_height_m'].min():.2f} m")
print("\n" + "="*70)
print("✅ ANÁLISIS COMPLETADO")
print("="*70)